# Notebook 4: Data Virtualization

**Pattern:** Federated queries across heterogeneous sources without ETL. Leave data where it lives.

**Stack:** DuckDB multi-source federation (portable proxy for Trino / watsonx.data federation). Query Postgres AND Parquet files without moving data.

**Maple Trust Bank** — synthetic BFSI data

🎯 **Cameo cell** — run Q1 during Block 1 for a quick federation demo (3 min).

---
## Configuration — Plan A / Plan B

Toggle the `USE_PLAN_A` flag below to switch between:
- **Plan A:** Trino or watsonx.data federation
- **Plan B:** DuckDB with `postgres_scanner` extension (local Docker)

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
USE_PLAN_A = False  # Set True for Trino/watsonx.data; False for DuckDB + postgres_scanner

# Postgres connection (Docker)
PG_HOST = "localhost"
PG_PORT = 5432
PG_USER = "lecture"
PG_PASSWORD = "lecture"
PG_DATABASE = "bfsi"

# Data paths (for parquet files — Source B)
DATA_DIR = "../data"
LINEAGE_PATH = f"{DATA_DIR}/lineage/lineage_graph.json"

if USE_PLAN_A:
    print("Plan A: Trino / watsonx.data federation — configure Trino endpoints.")
else:
    print("Plan B: DuckDB + postgres_scanner — ensure 'docker compose up postgres' is running.")

In [ ]:
import duckdb
import pandas as pd
import json
import time
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

---
## Section 1: The Pattern in One Paragraph

**Data virtualization** is the "don't move it, query it" pattern. Instead of extracting, transforming, and loading data into a central warehouse, you federate queries across databases, files, and APIs in place — without copying. It sounds elegant, and it is — for the right use cases. The most misunderstood pattern in data architecture, virtualization is not faster (it adds network hops), not cheaper (federation engines are complex), and not simpler (query planning across heterogeneous sources is hard). What it IS about: data that *cannot* move. Data sovereignty laws, contractual obligations, latency requirements, or system-of-record ownership rules that prohibit copying. When someone tells you "we'll just virtualize everything," ask them about their SLAs.

---
## Section 2: When You'd Use It, When You Wouldn't

| Use when | Don't use when |
|----------|----------------|
| Data cannot move (sovereignty, contracts, regulation) | High-volume analytical scans (performance degrades) |
| Federated queries across heterogeneous sources | Latency-sensitive workloads (< 100ms) |
| Real-time access to operational systems (read-only) | Data is all in one place already |
| Reducing data copies to avoid governance sprawl | You need repeatable, deterministic analytics |
| Cross-LOB queries where each LOB owns its data | Your data sources have unreliable uptime |
| Quick proof-of-concept without building pipelines | Production dashboards with strict SLAs |

---
## Section 3: The Setup — Two Separate Data Sources

- **Source A:** Postgres (Docker) — branches and customers
- **Source B:** Raw parquet files — transactions and accounts

Data stays where it lives. We federate across both with DuckDB.

In [ ]:
# Source A: Load branches and customers into Postgres
print("📊 Reference Architecture Swimlane: Data Sources")
print("   Loading reference data into Postgres (simulating an operational system)\n")

pg_url = f"postgresql://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DATABASE}"
engine = create_engine(pg_url)

branches_df = pd.read_parquet(f"{DATA_DIR}/branches.parquet")
customers_df = pd.read_parquet(f"{DATA_DIR}/customers.parquet")

branches_df.to_sql("branches", engine, if_exists="replace", index=False)
customers_df.to_sql("customers", engine, if_exists="replace", index=False)

# Verify
with engine.connect() as conn:
    b_count = conn.execute(text("SELECT COUNT(*) FROM branches")).scalar()
    c_count = conn.execute(text("SELECT COUNT(*) FROM customers")).scalar()

print(f"  Source A (Postgres): branches={b_count}, customers={c_count}")
print("  Source B (Parquet):  transactions, accounts (on disk)")

In [ ]:
# Set up DuckDB with postgres_scanner for federation
con = duckdb.connect()

# Install and load postgres_scanner extension
con.execute("INSTALL postgres;")
con.execute("LOAD postgres;")

# Attach Postgres as a federated source
con.execute(f"""
    ATTACH 'dbname={PG_DATABASE} user={PG_USER} password={PG_PASSWORD} host={PG_HOST} port={PG_PORT}'
    AS pg_source (TYPE POSTGRES, READ_ONLY);
""")

print("DuckDB connected with postgres_scanner.")
print("  Source A: pg_source.public.branches, pg_source.public.customers (Postgres)")
print(f"  Source B: {DATA_DIR}/transactions.parquet, {DATA_DIR}/accounts.parquet (files)")
print()
print("Data has NOT been moved. Queries will federate across both sources.")

In [ ]:
# Verify both sources are accessible
pg_tables = con.execute(
    "SELECT table_name FROM information_schema.tables WHERE table_schema = 'pg_source.public' OR table_catalog = 'pg_source'"
).fetchdf()
print("Postgres tables accessible via federation:")
pg_branches = con.execute("SELECT COUNT(*) as cnt FROM pg_source.public.branches").fetchone()[0]
pg_customers = con.execute("SELECT COUNT(*) as cnt FROM pg_source.public.customers").fetchone()[0]
print(f"  pg_source.public.branches:  {pg_branches} rows")
print(f"  pg_source.public.customers: {pg_customers:,} rows")

parquet_txn = con.execute(f"SELECT COUNT(*) FROM '{DATA_DIR}/transactions.parquet'").fetchone()[0]
parquet_acct = con.execute(f"SELECT COUNT(*) FROM '{DATA_DIR}/accounts.parquet'").fetchone()[0]
print("\nParquet files on disk:")
print(f"  transactions.parquet: {parquet_txn:,} rows")
print(f"  accounts.parquet:     {parquet_acct:,} rows")

---
## Section 4: Three Canonical Queries

### Q1: Total transaction volume by branch for Q3 2024 (FEDERATED)

🎯 **Cameo cell** — run this during Block 1 for a quick federation demo.

This query joins Postgres (branches) with parquet files (transactions) — data never moved.

In [ ]:
# FEDERATED QUERY: Postgres branches + Parquet transactions
start = time.time()

q1 = con.execute(f"""
    SELECT
        t.branch_id,
        b.name AS branch_name,
        b.region,
        COUNT(*)           AS txn_count,
        SUM(t.amount)      AS total_amount,
        AVG(t.amount)      AS avg_amount
    FROM '{DATA_DIR}/transactions.parquet' t
    JOIN pg_source.public.branches b ON t.branch_id = b.branch_id
    WHERE t.timestamp >= '2024-07-01' AND t.timestamp < '2024-10-01'
    GROUP BY t.branch_id, b.name, b.region
    ORDER BY total_amount DESC
""").fetchdf()

federated_ms = (time.time() - start) * 1000

print("📊 Reference Architecture Swimlane: Data Access")
print("   (Data Virtualization = federated query across sources)\n")
print(f"Q1: Transaction volume by branch — Q3 2024 (FEDERATED, {federated_ms:.0f} ms)")
print("  Source A: pg_source.public.branches (Postgres)")
print("  Source B: transactions.parquet (local file)")
print("  Data moved: ZERO bytes. Query crossed system boundaries.\n")
q1.head(10)

### Q2: Find all customers whose policy documents reference AML procedure X

Virtualization cannot federate over unstructured content.

In [ ]:
print("Q2: Policy document search — AML procedure references")
print("=" * 60)
print()
print("RESULT: Cannot be answered via federation.")
print()
print("Virtualization federates structured queries across databases and files.")
print("It cannot:")
print("  → Search inside PDF documents")
print("  → Perform full-text search across document stores")
print("  → Federate over unstructured content")
print()
print("You could federate a query TO a search engine (e.g., Elasticsearch),")
print("but the virtualization layer doesn't understand document semantics.")
print()
print("→ Addressed in Notebook 6 (RAG) with vector search.")

### Q3: Trace the lineage of branch_summary_quarterly back to source

Virtualization makes lineage HARDER — there's no materialized layer to instrument.

In [ ]:
print("Q3: Lineage in a virtualized environment")
print("=" * 60)
print()
print("The fundamental lineage challenge with virtualization:")
print()
print("  In a warehouse/lakehouse: data is materialized → you can track what went in.")
print("  In virtualization: data is NOT materialized → lineage is the QUERY PLAN.")
print()
print("Consider our Q1 federated query:")
print("  → Source A: Postgres branches table (owned by Branch Operations)")
print("  → Source B: Parquet transactions file (owned by Data Engineering)")
print("  → Result: branch_summary — but WHERE does this result live? Nowhere.")
print()
print("The lineage graph for virtualized queries is:")
print("  1. Ephemeral — it only exists during query execution")
print("  2. Cross-system — it spans multiple ownership boundaries")
print("  3. Non-deterministic — source data may change between executions")

In [ ]:
# Show the query plan to illustrate cross-source lineage
plan = con.execute(f"""
    EXPLAIN
    SELECT t.branch_id, b.name, COUNT(*) AS txn_count
    FROM '{DATA_DIR}/transactions.parquet' t
    JOIN pg_source.public.branches b ON t.branch_id = b.branch_id
    WHERE t.timestamp >= '2024-07-01' AND t.timestamp < '2024-10-01'
    GROUP BY t.branch_id, b.name
""").fetchdf()

print("Query plan (federation lineage):")
for row in plan.itertuples():
    print(row[1])
print()
print("⚠️  This plan IS the lineage. There's no persistent record unless you log it.")
print("   IBM Data Virtualization (CP4D) logs query plans for governance.")

In [ ]:
# For reference: the external lineage graph
with open(LINEAGE_PATH) as f:
    lineage = json.load(f)

target = "consumed.branch_summary_quarterly"


def trace_lineage(target_node, edges, depth=0):
    upstream = [e for e in edges if e["to"] == target_node]
    for edge in upstream:
        indent = "  " * depth
        print(f"{indent}← {edge['from']}")
        trace_lineage(edge["from"], edges, depth + 1)


print(f"External lineage graph for: {target}")
trace_lineage(target, lineage["edges"])
print()
print("In a virtualized world, each arrow in this graph is a live federation link.")
print("If any source goes down, the query fails. No cached fallback.")

---
## Section 5: Where This Pattern Breaks — Performance

In [ ]:
# Performance comparison: federated vs. local
print("Performance break: Federated vs. Local query on full 1M transactions\n")

# Federated: join across Postgres + Parquet
start = time.time()
federated_result = con.execute(f"""
    SELECT
        b.region,
        COUNT(*)           AS txn_count,
        SUM(t.amount)      AS total_amount
    FROM '{DATA_DIR}/transactions.parquet' t
    JOIN pg_source.public.branches b ON t.branch_id = b.branch_id
    GROUP BY b.region
    ORDER BY total_amount DESC
""").fetchdf()
federated_time = (time.time() - start) * 1000

print(f"  Federated (Postgres + Parquet): {federated_time:.0f} ms")
print(federated_result.to_string(index=False))

In [ ]:
# Local: same query on local parquet files only (no federation)
start = time.time()
local_result = con.execute(f"""
    SELECT
        b.region,
        COUNT(*)           AS txn_count,
        SUM(t.amount)      AS total_amount
    FROM '{DATA_DIR}/transactions.parquet' t
    JOIN '{DATA_DIR}/branches.parquet' b ON t.branch_id = b.branch_id
    GROUP BY b.region
    ORDER BY total_amount DESC
""").fetchdf()
local_time = (time.time() - start) * 1000

print(f"  Local (all Parquet):            {local_time:.0f} ms")
print(local_result.to_string(index=False))

In [ ]:
# Summary
slowdown = federated_time / local_time if local_time > 0 else float("inf")

print("\nPerformance comparison:")
print(f"  Federated: {federated_time:.0f} ms")
print(f"  Local:     {local_time:.0f} ms")
print(f"  Slowdown:  {slowdown:.1f}x")
print()
print("The federation overhead comes from:")
print("  1. Network round-trip to Postgres")
print("  2. Cross-engine query planning")
print("  3. Data serialization/deserialization across the boundary")
print()
print('"Virtualization is about ACCESS, not PERFORMANCE."')
print('"When someone tells you virtualization replaces ETL, ask them about their SLAs."')

---
## Section 6: The IBM Stack Mapping

| Component | IBM Product | Swimlane |
|-----------|-------------|----------|
| Federation Engine | Data Virtualization (CP4D) | Data Access |
| Federation Engine | watsonx.data (Presto federation) | Data Access |
| Connectors | 100+ connectors (JDBC, ODBC, REST) | Ingestion & Integration |
| Partner | Denodo (strategic partner) | Data Access |
| Governance | IBM Knowledge Catalog (query logging) | Information & Model Management & Governance |

**Swimlane:** Data Access — the middle-right section of the reference architecture diagram.

**Note:** "Data Virtualization is IBM's answer to 'we have 47 data sources and the CISO won't let us copy anything.'"

In [ ]:
print("📊 Reference Architecture Swimlane: Data Access")
print("   (middle-right section of the reference architecture)\n")
print("IBM Product Mapping:")
print("  Federation Engine → Data Virtualization service (CP4D)")
print("  Federation Engine → watsonx.data (Presto federation)")
print("  Connectors        → 100+ connectors (JDBC, ODBC, REST, S3, etc.)")
print("  Partner           → Denodo (strategic partner for complex federation)")
print("  Governance        → IBM Knowledge Catalog (query plan logging)")
print()
print("Data Virtualization is IBM's answer to:")
print("'We have 47 data sources and the CISO won't let us copy anything.'")

---
## Section 7: BFSI Reality Check

Canadian banks use virtualization heavily for cross-line-of-business queries where data sovereignty prevents consolidation. A retail banking analyst needs commercial banking exposure data for a relationship view, but the commercial data sits in a different system of record under a different data steward. OSFI (the regulator) requires certain data to remain in specific systems of record — the account master must stay in the core banking system, the KYC records must stay in the compliance platform. Virtualization doesn't replace these systems; it provides a read-only query layer across them. The banks that use it successfully treat it as what it is: a compliance access tool, not an analytics engine. The ones that fail try to build dashboards on federated queries and discover that join performance across system boundaries is unpredictable. Virtualization isn't an architecture choice — it's a regulatory compliance tool.

In [ ]:
# Clean up
con.close()
engine.dispose()
print("Notebook 4 complete.")
print()
print("Key takeaway: Virtualization is about access, not performance.")
print("Use it when data CANNOT move. Not when you CHOOSE not to move it.")
print("Next: Notebook 5 (Data Mesh) — what if the problem isn't technology at all?")